# High Volume Scanner — End of Day

Run **after 3:00 PM ET** (US) or **after 3:30 PM IST** (Nifty).

Finds stocks at their highest volume in 3 months / 1 year / 8 years.

**Universe toggles:** `INCLUDE_NIFTY` · `INCLUDE_SP500` · `INCLUDE_RUSSELL`

**🔬 Diagnostic tool** at the bottom — paste Chartink tickers to compare condition-by-condition.

**Runtime > Run all**


# AUTOMATION: *skip
# COLAB ONLY — download and run keepalive from the GitHub repository

import requests
from pathlib import Path

KEEPALIVE_URL = "https://raw.githubusercontent.com/guddu1221/stock-scanner/main/colab_keepalive.ipynb"

keepalive_ipynb = Path("/content/colab_keepalive.ipynb")
keepalive_py = Path("/content/colab_keepalive.py")

response = requests.get(KEEPALIVE_URL, timeout=30)
response.raise_for_status()
keepalive_ipynb.write_bytes(response.content)

print("✅ colab_keepalive.ipynb downloaded from GitHub")
    
!jupyter nbconvert --to script /content/colab_keepalive.ipynb --output /content/colab_keepalive.py

%run /content/colab_keepalive.py

print("✅ Colab keepalive loaded")

In [ ]:
# CELL 1 — Install
import subprocess, sys
import datetime
subprocess.check_call([sys.executable,"-m","pip","install","-q",
    "yfinance","tqdm","requests","lxml","gspread","gspread-dataframe","google-auth"])
print("Packages ready")


now = datetime.datetime.now()

print(f" Running cell 3 -> Current date and time :{now} ")

Packages ready


In [ ]:
# CELL 2 — Settings
BATCH_SIZE            = 40
SLEEP_BETWEEN_BATCHES = 2
SHEET_ID              = "1rzc_6fZoHMFi1Ee75zRmIuGxeCWog1E62pZp9c1Lsfs"

# ── Universe toggles ─────────────────────────────────────────
# Controlled by SCAN_MARKET env var (india / usa / both)
# Or override manually below after the auto-detection block
# ── Universe / market mode ──────────────────────────────────
# Set SCAN_MARKET env var to 'india' or 'usa' for scheduled automation.
# INDIA: scan Nifty 500 only.
# USA  : scan S&P 500 + Russell 2000 only.
# Default: 'both' (scans everything, same as before)
import os
SCAN_MARKET = os.environ.get("SCAN_MARKET", "both").strip().lower()
if SCAN_MARKET not in {"india", "usa", "both"}:
    raise ValueError(f"SCAN_MARKET must be 'india', 'usa', or 'both', got: {SCAN_MARKET!r}")

if SCAN_MARKET == "india":
    INCLUDE_NIFTY   = True
    INCLUDE_SP500   = False
    INCLUDE_RUSSELL = False
elif SCAN_MARKET == "usa":
    INCLUDE_NIFTY   = False
    INCLUDE_SP500   = True
    INCLUDE_RUSSELL = False   # set True if you want Russell too
else:  # both
    INCLUDE_NIFTY   = True
    INCLUDE_SP500   = True
    INCLUDE_RUSSELL = False   # keep off by default — slow

print(f"SCAN_MARKET : {SCAN_MARKET.upper()}")
print(f"  S&P 500   : {INCLUDE_SP500}")
print(f"  Russell   : {INCLUDE_RUSSELL}")
print(f"  Nifty 500 : {INCLUDE_NIFTY}")

print(f"Universe:")
print(f"  S&P 500        : {INCLUDE_SP500}")
print(f"  Russell 2000   : {INCLUDE_RUSSELL}")
print(f"  Nifty 500      : {INCLUDE_NIFTY}")
print(f"Batch size       : {BATCH_SIZE}")

now = datetime.datetime.now()

print(f" Running cell 2 Setting -> Current date and time :{now} ")

In [ ]:
# CELL 3 — Load tickers (S&P 500 + Russell 2000 + Nifty 500)
import requests, io
import pandas as pd
import numpy as np

BATCH_SIZE            = 40
SLEEP_BETWEEN_BATCHES = 2
HEADERS = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}

def get_sp500():
    try:
        html = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies", headers=HEADERS, timeout=15).text
        df = pd.read_html(io.StringIO(html))[0]
        tickers = [str(t).replace(".", "-") for t in df["Symbol"].tolist()]
        print(f"  S&P 500: {len(tickers)} tickers")
        return tickers
    except Exception as e:
        print(f"  S&P 500 failed: {e}")
        return []

def get_russell2000():
    """Build Russell 2000 proxy from NYSE + NASDAQ full lists minus S&P 500.
    More reliable than iShares CSV which blocks scraping.
    Gives ~1900-2000 small/mid cap US stocks — close to actual Russell 2000."""
    try:
        nyse_url = "https://raw.githubusercontent.com/rreichel3/US-Stock-Symbols/main/nyse/nyse_tickers.txt"
        nasd_url = "https://raw.githubusercontent.com/rreichel3/US-Stock-Symbols/main/nasdaq/nasdaq_tickers.txt"
        sp_url   = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv"
        nyse = requests.get(nyse_url, headers=HEADERS, timeout=15).text.strip().split()
        nasd = requests.get(nasd_url, headers=HEADERS, timeout=15).text.strip().split()
        # Keep only clean alpha tickers 1-5 chars (excludes warrants, rights, preferred)
        nyse_clean = [t for t in nyse if t.isalpha() and 1 <= len(t) <= 5]
        nasd_clean = [t for t in nasd if t.isalpha() and 1 <= len(t) <= 5]
        all_us = list(dict.fromkeys(nyse_clean + nasd_clean))
        # Remove S&P 500 tickers
        sp_df = pd.read_csv(io.StringIO(requests.get(sp_url, headers=HEADERS, timeout=15).text))
        sp500_set = set(sp_df["Symbol"].str.replace(".", "-", regex=False).tolist())
        tickers = [t for t in all_us if t not in sp500_set]
        print(f"  Russell 2000 proxy: {len(tickers)} tickers (NYSE+NASDAQ minus S&P500)")
        return tickers
    except Exception as e:
        print(f"  Russell 2000 failed: {e}")
        return []
def get_nifty500():
    # Official Nifty 500 NSE symbols — embedded, no HTTP needed
    symbols = [
        "360ONE","3MINDIA","ABB","ACC","ACMESOLAR","AIAENG","APLAPOLLO","AUBANK",
        "AWL","AADHARHFC","AARTIIND","AAVAS","ABBOTINDIA","ACE","ACUTAAS","ADANIENSOL",
        "ADANIENT","ADANIGREEN","ADANIPORTS","ADANIPOWER","ATGL","ABCAPITAL","ABFRL","ABLBL",
        "ABREL","ABSLAMC","CPPLUS","AEGISLOG","AEGISVOPAK","AFCONS","AFFLE","AJANTPHARM",
        "ALKEM","ABDL","AMBER","AMBUJACEM","ANANDRATHI","ANANTRAJ","ANGELONE","ANTHEM",
        "ANURAS","APARINDS","APOLLOHOSP","APOLLOTYRE","APTUS","ASAHIINDIA","ASHOKLEY","ASIANPAINT",
        "ASTERDM","ASTRAL","ATHERENERG","ATUL","AUROPHARMA","AIIL","DMART","AXISBANK",
        "BEML","BLS","BSE","BAJAJ-AUTO","BAJFINANCE","BAJAJFINSV","BAJAJHLDNG","BAJAJHFL",
        "BALKRISIND","BALRAMCHIN","BANDHANBNK","BANKBARODA","BANKINDIA","MAHABANK","BATAINDIA","BAYERCROP",
        "BELRISE","BERGEPAINT","BDL","BEL","BHARATFORG","BHEL","BPCL","BHARTIARTL",
        "BHARTIHEXA","BIKAJI","GROWW","BIOCON","BSOFT","BLUEDART","BLUEJET","BLUESTARCO",
        "BBTC","BOSCHLTD","FIRSTCRY","BRIGADE","BRITANNIA","MAPMYINDIA","CCL","CESC",
        "CGPOWER","CIEINDIA","CRISIL","CANFINHOME","CANBK","CANHLIFE","CAPLIPOINT","CGCL",
        "CARBORUNIV","CARTRADE","CASTROLIND","CEATLTD","CEMPRO","CENTRALBK","CDSL","CHALET",
        "CHAMBLFERT","CHENNPETRO","CHOICEIN","CHOLAHLDNG","CHOLAFIN","CIPLA","CUB","CLEAN",
        "COALINDIA","COCHINSHIP","COFORGE","COHANCE","COLPAL","CAMS","CONCORDBIO","CONCOR",
        "COROMANDEL","CRAFTSMAN","CREDITACC","CROMPTON","CUMMINSIND","CYIENT","DCMSHRIRAM","DLF",
        "DOMS","DABUR","DALBHARAT","DATAPATTNS","DEEPAKFERT","DEEPAKNTR","DELHIVERY","DEVYANI",
        "DIVISLAB","DIXON","LALPATHLAB","DRREDDY","EIDPARRY","EIHOTEL","EICHERMOT","ELECON",
        "ELGIEQUIP","EMAMILTD","EMCURE","EMMVEE","ENDURANCE","ENGINERSIN","ERIS","ESCORTS",
        "ETERNAL","EXIDEIND","NYKAA","FEDERALBNK","FACT","FINCABLES","FSL","FIVESTAR",
        "FORCEMOT","FORTIS","GAIL","GMRAIRPORT","GABRIEL","GALLANTT","GRSE","GICRE",
        "GILLETTE","GLAND","GLAXO","GLENMARK","MEDANTA","GODIGIT","GPIL","GODFRYPHLP",
        "GODREJCP","GODREJIND","GODREJPROP","GRANULES","GRAPHITE","GRASIM","GRAVITA","GESHIP",
        "FLUOROCHEM","GMDCLTD","HEG","HBLENGINE","HCLTECH","HDBFS","HDFCAMC","HDFCBANK",
        "HDFCLIFE","HFCL","HAVELLS","HEROMOTOCO","HEXT","HSCL","HINDALCO","HAL",
        "HINDCOPPER","HINDPETRO","HINDUNILVR","HINDZINC","POWERINDIA","HOMEFIRST","HONASA","HONAUT",
        "HUDCO","HYUNDAI","ICICIBANK","ICICIGI","ICICIAMC","ICICIPRULI","IDBI","IDFCFIRSTB",
        "IFCI","IIFL","IRB","IRCON","ITCHOTELS","ITC","ITI","INDGN",
        "INDIACEM","INDIAMART","INDIANB","IEX","INDHOTEL","IOC","IOB","IRCTC",
        "IRFC","IREDA","IGL","INDUSTOWER","INDUSINDBK","NAUKRI","INFY","INOXWIND",
        "INTELLECT","INDIGO","IGIL","IKS","IPCALAB","JBCHEPHARM","JKCEMENT","JBMA",
        "JKTYRE","JMFINANCIL","JSWCEMENT","JSWDULUX","JSWENERGY","JSWINFRA","JSWSTEEL","JAINREC",
        "JPPOWER","JINDALSAW","JSL","JINDALSTEL","JIOFIN","JUBLFOOD","JUBLINGREA","JUBLPHARMA",
        "JWL","JYOTICNC","KPRMILL","KEI","KPITTECH","KAJARIACER","KPIL","KALYANKJIL",
        "KARURVYSYA","KAYNES","KEC","KFINTECH","KIRLOSENG","KOTAKBANK","KIMS","LTF",
        "LTTS","LGEINDIA","LICHSGFIN","LTFOODS","LTM","LT","LATENTVIEW","LAURUSLABS",
        "THELEELA","LEMONTREE","LENSKART","LICI","LINDEINDIA","LLOYDSME","LODHA","LUPIN",
        "MMTC","MRF","MGL","M&MFIN","M&M","MANAPPURAM","MRPL","MANKIND",
        "MARICO","MARUTI","MFSL","MAXHEALTH","MAZDOCK","MEESHO","MINDACORP","MSUMI",
        "MOTILALOFS","MPHASIS","MCX","MUTHOOTFIN","NATCOPHARM","NBCC","NCC","NHPC",
        "NLCINDIA","NMDC","NSLNISP","NTPCGREEN","NTPC","NH","NATIONALUM","NAVA",
        "NAVINFLUOR","NESTLEIND","NETWEB","NEULANDLAB","NEWGEN","NAM-INDIA","NIVABUPA","NUVAMA",
        "NUVOCO","OBEROIRLTY","ONGC","OIL","OLAELEC","OLECTRA","PAYTM","ONESOURCE",
        "OFSS","POLICYBZR","PCBL","PGEL","PIIND","PNBHOUSING","PTCIL","PVRINOX",
        "PAGEIND","PARADEEP","PATANJALI","PERSISTENT","PETRONET","PFIZER","PHOENIXLTD","PWL",
        "PIDILITIND","PINELABS","PIRAMALFIN","PPLPHARMA","POLYMED","POLYCAB","POONAWALLA","PFC",
        "POWERGRID","PREMIERENE","PRESTIGE","PNB","RRKABEL","RBLBANK","RECLTD","RHIM",
        "RITES","RADICO","RVNL","RAILTEL","RAINBOW","RKFORGE","REDINGTON","RELIANCE",
        "RPOWER","SBFC","SBICARD","SBILIFE","SJVN","SRF","SAGILITY","SAILIFE",
        "SAMMAANCAP","MOTHERSON","SAPPHIRE","SARDAEN","SAREGAMA","SCHAEFFLER","SCHNEIDER","SCI",
        "SHREECEM","SHRIRAMFIN","SHYAMMETL","ENRIN","SIEMENS","SIGNATURE","SOBHA","SOLARINDS",
        "SONACOMS","SONATSOFTW","STARHEALTH","SBIN","SAIL","SUMICHEM","SUNPHARMA","SUNTV",
        "SUNDARMFIN","SUPREMEIND","SPLPETRO","SUZLON","SWANCORP","SWIGGY","SYNGENE","SYRMA",
        "TBOTEK","TVSMOTOR","TATACAP","TATACHEM","TATACOMM","TCS","TATACONSUM","TATAELXSI",
        "TATAINVEST","TMCV","TMPV","TATAPOWER","TATASTEEL","TATATECH","TTML","TECHM",
        "TECHNOE","TEGA","TEJASNET","TENNIND","NIACL","RAMCOCEM","THERMAX","TIMKEN",
        "TITAGARH","TITAN","TORNTPHARM","TORNTPOWER","TARIL","TRAVELFOOD","TRENT","TRIDENT",
        "TRITURBINE","TIINDIA","UCOBANK","UNOMINDA","UPL","UTIAMC","ULTRACEMCO","UNIONBANK",
        "UBL","UNITDSPR","URBANCO","USHAMART","VTL","VBL","VEDL","VIJAYA",
    ]
    symbols = list(dict.fromkeys(symbols))
    tickers = [s + ".NS" for s in symbols]
    print(f"  Nifty 500: {len(tickers)} tickers")
    return tickers
sp500    = get_sp500()    if INCLUDE_SP500   else []
r2000    = get_russell2000() if INCLUDE_RUSSELL else []
nifty500 = get_nifty500()   if INCLUDE_NIFTY   else []

# Russell 2000 excludes S&P 500 to avoid duplicates
r2000_only = [t for t in r2000 if t not in set(sp500)] if r2000 else []

total = len(sp500) + len(r2000_only) + len(nifty500)
print(f"  S&P 500        : {len(sp500)}")
print(f"  Russell 2000   : {len(r2000_only)} (unique, excl S&P500)")
print(f"  Nifty 500      : {len(nifty500)}")
print(f"  Total universe : {total}")
now = datetime.datetime.now()

print(f" Running cell load ticker-> Current date and time :{now} ")

In [ ]:
# CELL 5 — High Volume scan function
# Exact logic from working file — auto_adjust=True compatible
import pandas as pd
import numpy as np

def _sma(s, n): return s.rolling(n).mean()

def _scan_hv(d, w, mktcap_m):
    """
    Group A (any 1): today vol = rolling max over 63d / 252d / 2000d
    Group B (all):   close>=20, mktcap>=100, wk_sma_vol>100k,
                     sma50vol*close>5M, close>=prev_close

    mktcap_m units:
      US stocks  : USD millions  (price_usd * avg_vol * 30 / 1e6)
      Nifty (.NS): INR crores    (price_inr * avg_vol * 30 / 1e7)
    Both use threshold >= 100 which means:
      US   : $100M market cap  ✓ filters micro caps
      Nifty: 100 crores (~$12M) ✓ filters very small stocks

    Dollar/Rupee volume threshold 5,000,000:
      US   : $5M daily dollar volume ✓
      Nifty: ₹5M = ₹50 lakh daily ✓ (low but NSE volumes are high)
    """
    try:
        if len(d) < 63 or len(w) < 12: return False
        vol        = float(d["Volume"].iloc[-1])
        close      = float(d["Close"].iloc[-1])
        prev_close = float(d["Close"].iloc[-2])
        if vol <= 0 or close <= 0: return False

        # ── Group A: volume record ─────────────────────────────
        lookback = min(len(d), 2000)
        max_2000 = d["Volume"].rolling(lookback, min_periods=lookback).max().iloc[-1]
        max_252  = d["Volume"].rolling(min(252, len(d)), min_periods=252).max().iloc[-1]
        max_63   = d["Volume"].rolling(63, min_periods=63).max().iloc[-1]

        group_a = (
            (pd.notna(max_2000) and vol >= float(max_2000)) or
            (pd.notna(max_252)  and vol >= float(max_252))  or
            (pd.notna(max_63)   and vol >= float(max_63))
        )
        if not group_a: return False

        # ── Group B: quality filters ───────────────────────────
        avg_wk_vol = _sma(w["Volume"], 10).iloc[-1]
        avg_vol_50 = _sma(d["Volume"], 50).iloc[-1]
        if pd.isna(avg_wk_vol) or pd.isna(avg_vol_50): return False

        return (
            close >= 20
            and mktcap_m >= 100
            and float(avg_wk_vol) > 100000
            and float(avg_vol_50) * close > 5000000
            and close >= prev_close
        )
    except:
        return False

print("Scan function loaded")
print("mktcap: USD millions for US (>=100 = $100M), INR crores for Nifty (>=100 = 100Cr)")
now = datetime.datetime.now()

print(f" Running cell High Volume scan -> Current date and time :{now} ")

Scan function loaded
mktcap: USD millions for US (>=100 = $100M), INR crores for Nifty (>=100 = 100Cr)


In [ ]:
# CELL 6 — RUN HIGH VOLUME SCAN
# Run after 3:00 PM ET (US) | after 3:30 PM IST (Nifty)
import time
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime

def _fetch(tickers, period, interval):
    """
    Same as original _fetch() but with:
    - Retry up to 3 times on rate limit (exponential backoff)
    - Correct MultiIndex handling for both single and multi-ticker
    - Skips empty/bad data silently
    """
    out = {}
    if not tickers: return out

    for attempt in range(3):
        try:
            raw = yf.download(tickers, period=period, interval=interval,
                              group_by="ticker", auto_adjust=True,
                              progress=False, threads=True)
            if raw.empty:
                return out

            if len(tickers) == 1:
                df = raw.copy()
                # Flatten MultiIndex columns if present
                if isinstance(df.columns, pd.MultiIndex):
                    df.columns = [c[0] for c in df.columns]
                df.dropna(how="all", inplace=True)
                if len(df) > 5:
                    out[tickers[0]] = df

            else:
                # Multi-ticker: yfinance returns columns as (field, ticker)
                for t in tickers:
                    try:
                        # Try level=1 (ticker is second level)
                        if isinstance(raw.columns, pd.MultiIndex):
                            lvl1 = raw.columns.get_level_values(1)
                            lvl0 = raw.columns.get_level_values(0)
                            if t in lvl1:
                                df = raw.xs(t, axis=1, level=1).copy()
                            elif t in lvl0:
                                df = raw.xs(t, axis=1, level=0).copy()
                            else:
                                continue
                        else:
                            df = raw[[t]].copy() if t in raw.columns else pd.DataFrame()
                        df.dropna(how="all", inplace=True)
                        if len(df) > 5:
                            out[t] = df
                    except:
                        pass

            return out  # success — exit retry loop

        except Exception as e:
            err = str(e)
            if "429" in err or "Rate" in err or "Too Many" in err:
                wait = (2 ** attempt) * 15  # 15s, 30s, 60s
                print(f"  Rate limited (attempt {attempt+1}/3) — waiting {wait}s")
                time.sleep(wait)
            else:
                # Non-rate-limit error — don't retry
                print(f"  Fetch error: {err[:60]}")
                return out

    print(f"  Max retries reached for batch of {len(tickers)}")
    return out

def run_volume_scan(tickers, label):
    if not tickers:
        print(f"{label} — skipped (toggle is OFF)")
        return []
    is_nifty = label == "Nifty 500"
    hits = []
    batches = [tickers[i:i+BATCH_SIZE] for i in range(0, len(tickers), BATCH_SIZE)]
    total = len(batches)
    print(f"{label} — {len(tickers)} tickers, {total} batches")
    for n, batch in enumerate(batches):
        print(f"  Batch {n+1}/{total} ...", end="")
        daily  = _fetch(batch, "max", "1d")
        weekly = _fetch(batch, "5y",  "1wk")
        for t in batch:
            d = daily.get(t); w = weekly.get(t)
            if d is None or len(d) < 63: continue
            try:
                mktcap = (float(d["Close"].iloc[-1]) *
                          float(_sma(d["Volume"], 20).iloc[-1]) *
                          30 / (1e7 if is_nifty else 1e6))
            except:
                mktcap = 0
            if w is not None and _scan_hv(d, w, mktcap):
                hits.append(t)
        time.sleep(SLEEP_BETWEEN_BATCHES)
    print(f"\n{label} done — {len(hits)} matches")
    return hits

start = datetime.now()
print(f"Started: {start.strftime('%H:%M:%S')}")
print()

sp500_hits  = run_volume_scan(sp500,      "S&P 500")     if INCLUDE_SP500   else []
r2000_hits  = run_volume_scan(r2000_only, "Russell 2000") if INCLUDE_RUSSELL else []
nifty_hits  = run_volume_scan(nifty500,   "Nifty 500")   if INCLUDE_NIFTY   else []

elapsed = (datetime.now() - start).seconds // 60
print(f"\nAll done in ~{elapsed} min")
print(f"S&P 500      : {len(sp500_hits)}  {', '.join(sp500_hits[:10])}")
print(f"Russell 2000 : {len(r2000_hits)}  {', '.join(r2000_hits[:10])}")
print(f"Nifty 500    : {len(nifty_hits)}  {', '.join(nifty_hits[:10])}")

# ── Save to Google Drive for Streamlit website ──────────────────
import os, pandas as pd
from datetime import datetime

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    _save_dir = '/content/drive/MyDrive/Stockbee'
    os.makedirs(_save_dir, exist_ok=True)
except Exception:
    _save_dir = '.'

_ts   = datetime.now().strftime('%Y-%m-%d %H:%M')
_rows = (
    [{'Ticker':t,'Universe':'S&P 500',   'Updated':_ts} for t in sp500_hits] +
    [{'Ticker':t,'Universe':'Russell 2000','Updated':_ts} for t in r2000_hits] +
    [{'Ticker':t,'Universe':'Nifty 500', 'Updated':_ts} for t in nifty_hits]
)
_eod_df  = pd.DataFrame(_rows) if _rows else pd.DataFrame(columns=['Ticker','Universe','Updated'])
_csv_path = os.path.join(_save_dir, 'eod_hits.csv')
_eod_df.to_csv(_csv_path, index=False)
print(f"\n✅ Saved eod_hits.csv → {_csv_path}  ({len(_eod_df)} rows)")
print(f"   S&P: {len(sp500_hits)}  R2K: {len(r2000_hits)}  Nifty: {len(nifty_hits)}")

now = datetime.datetime.now()

print(f" Running cell high volume scan cell 6 -> Current date and time :{now} ")

In [ ]:
# CELL 7 — Save CSVs and publish to Google Sheets
import subprocess, sys
subprocess.check_call([sys.executable,"-m","pip","install","-q",
    "gspread","gspread-dataframe","google-auth"])

import pandas as pd
import yfinance as yf
# import gspread
# from gspread_dataframe import set_with_dataframe
# from google.colab import auth, files
from datetime import datetime

def _sma(s, n): return s.rolling(n).mean()

ts        = datetime.now().strftime("%Y-%m-%d %H:%M")
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

all_hits = (
    [{"Ticker":t,"Universe":"S&P 500"}     for t in sp500_hits]  +
    [{"Ticker":t,"Universe":"Russell 2000"} for t in r2000_hits] +
    [{"Ticker":t,"Universe":"Nifty 500"}    for t in nifty_hits]
)
print(f"S&P 500      : {len(sp500_hits)}")
print(f"Russell 2000 : {len(r2000_hits)}")
print(f"Nifty 500    : {len(nifty_hits)}")
# print(f"Total        : {len(all_hits)}")

# df_out = pd.DataFrame(all_hits) if all_hits else pd.DataFrame(columns=["Ticker","Universe"])
# fname = f"high_volume_{timestamp}.csv"
# df_out.to_csv(fname, index=False)
# files.download(fname)
# print(f"Downloaded: {fname}")

# detail_rows = []
# for row in all_hits:
#     t = row["Ticker"]
#     try:
#         d = yf.download(t, period="1y", interval="1d",
#                         auto_adjust=True, progress=False)
#         if isinstance(d.columns, pd.MultiIndex):
#             d.columns = [c[0] for c in d.columns]
#         d.dropna(how="all", inplace=True)
#         if len(d) < 20: continue
#         vol   = int(d["Volume"].iloc[-1])
#         avg20 = int(_sma(d["Volume"],20).iloc[-1])
#         avg50 = int(_sma(d["Volume"],50).iloc[-1])
#         close = round(float(d["Close"].iloc[-1]),2)
#         chg   = round((close/float(d["Close"].iloc[-2])-1)*100,2)
#         detail_rows.append({"Ticker":t,"Universe":row["Universe"],
#             "Close":close,"Chg%":chg,"Volume":vol,
#             "AvgVol20":avg20,"AvgVol50":avg50,
#             "Vol_Avg20":round(vol/avg20,1) if avg20>0 else 0})
#     except: pass

# detail_df = pd.DataFrame(detail_rows).sort_values("Vol_Avg20",ascending=False) if detail_rows else pd.DataFrame()
# if not detail_df.empty:
#     df2 = f"high_volume_detail_{timestamp}.csv"
#     detail_df.to_csv(df2, index=False)
#     files.download(df2)
#     print(f"Downloaded: {df2}")

# auth.authenticate_user()
# from google.auth import default
# from google.auth.transport.requests import Request
# creds, _ = default(scopes=[
#     "https://www.googleapis.com/auth/spreadsheets",
#     "https://www.googleapis.com/auth/drive",
# ])
# try: creds.refresh(Request())
# except: pass
# gc_client = gspread.authorize(creds)
# sh = gc_client.open_by_key(SHEET_ID)

# def write_tab(df, name):
#     name=name[:30]
#     try: ws=sh.worksheet(name); ws.clear()
#     except gspread.WorksheetNotFound: ws=sh.add_worksheet(title=name,rows=3000,cols=20)
#     if not df.empty: set_with_dataframe(ws,df)
#     print(f"  Written: {name} ({len(df)} rows)")

# write_tab(df_out, "High Volume")
# if not detail_df.empty:
#     write_tab(detail_df, "High Volume Detail")

# try:
#     ws_sum = sh.worksheet("Summary")
#     existing = pd.DataFrame(ws_sum.get_all_records())
#     existing = existing[~existing["Scan"].str.contains("High Volume",na=False)]
#     new_row = pd.DataFrame([{
#         "Scan":"High Volume EOD",
#         "SP500":len(sp500_hits),
#         "R2K":len(r2000_hits),
#         "Nifty":len(nifty_hits),
#         "Total":len(all_hits),
#         "SP500_tickers":", ".join(sp500_hits),
#         "R2K_tickers":", ".join(r2000_hits),
#         "Nifty_tickers":", ".join(nifty_hits),
#         "Updated":ts,
#     }])
#     updated = pd.concat([existing,new_row],ignore_index=True)
#     ws_sum.clear(); set_with_dataframe(ws_sum,updated)
#     print("  Updated: Summary")
# except Exception as e:
#     print(f"  Summary update skipped: {e}")

# print(f"\nPublished at {ts}")
# print(f"Sheet: https://docs.google.com/spreadsheets/d/{SHEET_ID}")
now = datetime.datetime.now()

print("Cell 8 completed. Google Sheets publishing disabled.")

print(f" Running cell save and publish to google sheet -> Current date and time :{now} ")

---


---
## 🔬 High Volume Scan — Diagnostic Tool

Paste tickers from Chartink below (NSE stocks: add `.NS` suffix, e.g. `RELIANCE.NS`).

The cell will check **every individual condition** of the High Volume scan and show:
- ✅ PASS / ❌ FAIL for each condition with the **actual value vs threshold**
- 🟡 NaN warnings — where Yahoo Finance has **missing data**
- 📊 Full OHLCV data quality report for each timeframe
- 📈 Volume history table (today vs 20d/50d/63d/252d averages)

This lets you **compare directly with Chartink** to find exactly where the mismatch is.


In [ ]:
# @title 🔬 HIGH VOLUME SCAN DIAGNOSTIC
# ─────────────────────────────────────────────────────────────────────────────
# PASTE YOUR CHARTINK TICKERS HERE
# NSE stocks: add .NS suffix   e.g. RELIANCE.NS, INFY.NS
# US stocks:  no suffix needed e.g. AAPL, NVDA
# Separate by comma or new line
CHARTINK_TICKERS = """
RELIANCE.NS
INFY.NS
TCS.NS
AAPL
"""
# ─────────────────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

# Parse ticker list
DEBUG_TICKERS = [
    t.strip().upper() for t in CHARTINK_TICKERS.replace(',', '\n').split('\n')
    if t.strip()
]
# Auto-add .NS for known NSE format (no dot, not a US ticker)
# — leave as-is; user should add .NS manually for clarity

# ── Terminal colours ──
G  = '\033[92m';  R = '\033[91m';  Y = '\033[93m'
B  = '\033[94m';  W = '\033[1m';   D = '\033[2m';  NC = '\033[0m'

def tick(v):   return f'{G}✅ PASS{NC}' if v else f'{R}❌ FAIL{NC}'
def nantick(v): return f'{Y}🟡 NaN {NC}' if (v is None or (isinstance(v,float) and np.isnan(v))) else tick(bool(v))
def fmt(v, spec='.2f'):
    if v is None or (isinstance(v, float) and np.isnan(v)): return f'{Y}NaN{NC}'
    try:    return f'{B}{v:{spec}}{NC}'
    except: return f'{B}{v}{NC}'

def _sma_val(s, n):
    try: return float(s.rolling(n).mean().iloc[-1])
    except: return float('nan')

def nan_report(df, label, min_rows=0):
    if df is None or df.empty:
        print(f'    {Y}⚠  {label}: EMPTY — no data returned from Yahoo Finance{NC}')
        return
    nans = df.isnull().sum(); nans = nans[nans > 0]
    row_status = f'{len(df)} rows'
    if min_rows and len(df) < min_rows:
        row_status += f' {Y}(need {min_rows}){NC}'
    if nans.empty:
        print(f'    {G}✅ {label}: clean — {row_status}{NC}')
    else:
        print(f'    {Y}⚠  {label}: NaNs in {dict(nans)} — {row_status}{NC}')
    # Show last 5 rows
    print(f'    {D}Last 5 rows:{NC}')
    with pd.option_context('display.max_columns', 8, 'display.width', 110,
                           'display.float_format', '{:.2f}'.format):
        for line in df[['Open','High','Low','Close','Volume']].tail(5).to_string().split('\n'):
            print(f'      {D}{line}{NC}')

# ── Main diagnostic loop ──────────────────────────────────────────────────
print(f'{W}{'='*70}{NC}')
print(f'{W}  🔬 HIGH VOLUME SCAN DIAGNOSTIC  —  {len(DEBUG_TICKERS)} tickers{NC}')
print(f'{W}  Scan logic: Group A (vol record) + Group B (quality filters){NC}')
print(f'{W}{'='*70}{NC}')

for TICKER in DEBUG_TICKERS:
    is_nifty = TICKER.endswith('.NS')
    print(f'\n{W}{'─'*70}{NC}')
    print(f'{W}  📌  {TICKER}  ({"NSE / Nifty" if is_nifty else "US Stock"}){NC}')
    print(f'{W}{'─'*70}{NC}')

    # ── Fetch ──
    print(f'\n  {D}Fetching daily (max) + weekly (5y)...{NC}')
    try:
        tk = yf.Ticker(TICKER)
        d  = tk.history(period='max', interval='1d',  auto_adjust=True)
        w  = tk.history(period='5y',  interval='1wk', auto_adjust=True)
    except Exception as e:
        print(f'  {R}❌ FETCH FAILED: {e}{NC}'); continue

    # ── Data quality report ──
    print(f'\n  {W}📊 DATA QUALITY{NC}')
    nan_report(d, 'Daily  (max period)', min_rows=63)
    nan_report(w, 'Weekly (5y)',         min_rows=12)

    if d is None or d.empty or len(d) < 63:
        print(f'  {R}❌ Insufficient daily data — cannot evaluate scan{NC}'); continue

    # ── Compute all values ──
    try:
        vol        = float(d['Volume'].iloc[-1])
        close      = float(d['Close'].iloc[-1])
        prev_close = float(d['Close'].iloc[-2])
        date_today = str(d.index[-1].date()) if hasattr(d.index[-1], 'date') else str(d.index[-1])

        # Rolling volume maxima
        lookback   = min(len(d), 2000)
        max_2000   = d['Volume'].rolling(lookback, min_periods=lookback).max().iloc[-1]
        max_252    = d['Volume'].rolling(min(252, len(d)), min_periods=252).max().iloc[-1]
        max_63     = d['Volume'].rolling(63, min_periods=63).max().iloc[-1]

        # Volume averages
        avg20      = _sma_val(d['Volume'], 20)
        avg50      = _sma_val(d['Volume'], 50)
        avg10wk    = _sma_val(w['Volume'], 10) if w is not None and len(w) >= 10 else float('nan')

        # Market cap proxy
        mktcap_m   = close * avg20 * 30 / (1e7 if is_nifty else 1e6)
        mktcap_unit = 'Crores (INR)' if is_nifty else 'Million (USD)'

        # Dollar/Rupee volume
        dv50       = avg50 * close

        # Volume ratios
        ratio_20   = vol / avg20  if avg20 > 0 else float('nan')
        ratio_50   = vol / avg50  if avg50 > 0 else float('nan')
        ratio_63   = vol / float(max_63)  if pd.notna(max_63)  and float(max_63)  > 0 else float('nan')
        ratio_252  = vol / float(max_252) if pd.notna(max_252) and float(max_252) > 0 else float('nan')
        ratio_2000 = vol / float(max_2000)if pd.notna(max_2000)and float(max_2000)> 0 else float('nan')

    except Exception as e:
        print(f'  {R}❌ Compute error: {e}{NC}'); continue

    # ── Volume history table ──
    print(f'\n  {W}📈 VOLUME SNAPSHOT  (as of {date_today}){NC}')
    print(f'    {D}Close           : {close:>15,.2f}{NC}')
    print(f'    {D}Prev Close      : {prev_close:>15,.2f}{NC}')
    rows_d = len(d)
    vol_history = [
        ('Today Vol',       vol,          ''),
        ('Avg Vol 20d',     avg20,        f'  ratio today/avg20 = {fmt(ratio_20)}'),
        ('Avg Vol 50d',     avg50,        f'  ratio today/avg50 = {fmt(ratio_50)}'),
        ('Max Vol 63d',     float(max_63)  if pd.notna(max_63)  else float('nan'), f'  today/max63  = {fmt(ratio_63)}'),
        ('Max Vol 252d',    float(max_252) if pd.notna(max_252) else float('nan'), f'  today/max252 = {fmt(ratio_252)}'),
        (f'Max Vol {rows_d}d (all)',float(max_2000)if pd.notna(max_2000)else float('nan'), f'  today/maxAll = {fmt(ratio_2000)}'),
        ('Avg Wk Vol 10wk', avg10wk,      ''),
        (f'DolVol 50d (close×avg50)', dv50, ''),
        (f'Mkt Cap proxy ({mktcap_unit})', mktcap_m, ''),
    ]
    for lbl, v, note in vol_history:
        vstr = f'{v:>20,.0f}' if not np.isnan(v) else f'{Y}  {"NaN":>18}{NC}'
        print(f'    {D}{lbl:<35}{NC}{B}{vstr}{NC}  {D}{note}{NC}')

    # Last 10 days volume
    print(f'\n  {W}📅 LAST 10 DAYS VOLUME{NC}')
    last10 = d[['Close','Volume']].tail(10).copy()
    last10['AvgVol20'] = d['Volume'].rolling(20).mean().tail(10).values
    last10['Ratio']    = (last10['Volume'] / last10['AvgVol20']).round(2)
    last10.index       = [str(i.date()) if hasattr(i,'date') else str(i) for i in last10.index]
    with pd.option_context('display.float_format', '{:,.0f}'.format, 'display.width', 110):
        for line in last10.to_string().split('\n'):
            print(f'    {D}{line}{NC}')

    # ── GROUP A: Volume record check ──
    print(f'\n  {W}📋 GROUP A — Volume Record (need ANY 1 of 3){NC}')
    a1 = pd.notna(max_63)  and vol >= float(max_63)
    a2 = pd.notna(max_252) and vol >= float(max_252)
    a3 = pd.notna(max_2000)and vol >= float(max_2000)
    group_a = a1 or a2 or a3
    for cond, desc, extra in [
        (a1, 'vol >= max vol in last 63 days  (3-month high)',
             f'today={fmt(vol,".0f")}  max63={fmt(float(max_63) if pd.notna(max_63) else float("nan"),".0f")}'),
        (a2, 'vol >= max vol in last 252 days (1-year high)',
             f'today={fmt(vol,".0f")}  max252={fmt(float(max_252) if pd.notna(max_252) else float("nan"),".0f")}'),
        (a3, f'vol >= max vol in last {rows_d} days (all-time high in data)',
             f'today={fmt(vol,".0f")}  maxAll={fmt(float(max_2000) if pd.notna(max_2000) else float("nan"),".0f")}'),
    ]:
        print(f'    {tick(cond)}  {desc}')
        print(f'          {D}{extra}{NC}')
    print(f'    {W}GROUP A RESULT: {tick(group_a)}{NC}')

    # ── GROUP B: Quality filters ──
    print(f'\n  {W}📋 GROUP B — Quality Filters (need ALL){NC}')
    b1 = close >= 20
    b2 = mktcap_m >= 100
    b3 = not np.isnan(avg10wk) and avg10wk > 100_000
    b4 = not np.isnan(avg50)   and avg50 * close > 5_000_000
    b5 = close >= prev_close
    group_b = b1 and b2 and b3 and b4 and b5
    for cond, desc, extra in [
        (b1, 'close >= 20',
             f'close={fmt(close)}'),
        (b2, f'mktcap proxy >= 100 {mktcap_unit}',
             f'mktcap={fmt(mktcap_m)} {mktcap_unit}  (formula: close × avg20vol × 30 / {"1e7" if is_nifty else "1e6"})'),
        (b3, 'weekly avg vol (10wk) > 100,000',
             f'avg10wk={fmt(avg10wk,".0f")}'),
        (b4, 'avg vol 50d × close > 5,000,000  (liquidity check)',
             f'avg50={fmt(avg50,".0f")}  close={fmt(close)}  product={fmt(dv50,".0f")}'),
        (b5, 'close >= prev close  (not down day)',
             f'close={fmt(close)}  prev_close={fmt(prev_close)}'),
    ]:
        print(f'    {tick(cond)}  {desc}')
        print(f'          {D}{extra}{NC}')
    print(f'    {W}GROUP B RESULT: {tick(group_b)}{NC}')

    # ── Final verdict ──
    final = group_a and group_b
    verdict_col = G if final else R
    print(f'\n  {verdict_col}{'─'*50}{NC}')
    print(f'  {verdict_col}  FINAL VERDICT: {"✅ WOULD HIT" if final else "❌ WOULD NOT HIT"}  (Group A {"✅" if group_a else "❌"} AND Group B {"✅" if group_b else "❌"}){NC}')
    if not final:
        failing = []
        if not group_a: failing.append('Group A: volume is NOT at a 63d/252d/all-time record')
        if not group_b:
            for cond, lbl in [(b1,'close<20'),(b2,'mktcap<100'),(b3,'wk_vol<=100K'),(b4,'dv50<=5M'),(b5,'down day')]:
                if not cond: failing.append(f'Group B: {lbl}')
        print(f'  {R}  Failing conditions:{NC}')
        for f_ in failing:
            print(f'    {R}→ {f_}{NC}')
    print(f'  {verdict_col}{'─'*50}{NC}')

print(f'\n{W}{'='*70}{NC}')
print(f'{W}  Diagnostic complete.{NC}')
print(f'{W}  ❌ FAIL rows + NaN values = mismatch vs Chartink{NC}')
print(f'{W}  If Group A fails but Chartink shows a hit:{NC}')
print(f'{W}    → Yahoo may have different volume data than NSE/BSE{NC}')
print(f'{W}    → Check "Last 10 days" table vs Chartink chart{NC}')
print(f'{W}{'='*70}{NC}')
now = datetime.datetime.now()

print(f" Running cell diagnostic -> Current date and time :{now} ")

## 🆕 New EOD Scans
Weekly RSI Reversal, BTST, Trend Reversal, Weekly Engulfing

In [ ]:

# ── NEW EOD SCANS ─────────────────────────────────────────────────────

def scan_weekly_rsi_reversal(d, w):
    """
    Chartink: Weekly RSI(14) > 50 AND daily SMA10 < close (price above SMA10)
    AND 5 days ago SMA10 > 5 days ago close (was below before)
    AND yesterday red candle AND today green candle
    Minimum market cap 5000 Cr.
    """
    try:
        if w is None or len(w) < 15 or len(d) < 15: return False
        # Weekly RSI > 50
        close_w = w['Close']
        def _rsi(s, n=14):
            delta = s.diff()
            gain  = delta.clip(lower=0).ewm(alpha=1/n, adjust=False).mean()
            loss  = (-delta.clip(upper=0)).ewm(alpha=1/n, adjust=False).mean()
            return 100 - (100 / (1 + gain / loss.replace(0, float('nan'))))
        w_rsi = _f(_rsi(close_w, 14).iloc[-1])
        if w_rsi <= 50: return False
        # Daily: SMA10 < close today
        sma10 = _f(d['Close'].rolling(10).mean().iloc[-1])
        if not (_f(d['Close'].iloc[-1]) > sma10): return False
        # 5 days ago SMA10 > 5 days ago close
        sma10_5ago  = _f(d['Close'].rolling(10).mean().iloc[-6])
        close_5ago  = _f(d['Close'].iloc[-6])
        if not (sma10_5ago > close_5ago): return False
        # Yesterday red, today green
        if not (_f(d['Close'].iloc[-2]) < _f(d['Open'].iloc[-2])): return False
        if not (_f(d['Close'].iloc[-1]) > _f(d['Open'].iloc[-1])): return False
        return True
    except: return False


def scan_btst_strong(d):
    """
    BTST (Buy Today Sell Tomorrow):
    close > EMA100 AND volume > 500K AND close > 100
    AND RSI(14) > 55 AND close > 1d ago high
    AND volume > SMA(vol,10) * 2
    """
    try:
        if len(d) < 15: return False
        ema100  = _f(d['Close'].ewm(span=100, adjust=False).mean().iloc[-1])
        vol     = _f(d['Volume'].iloc[-1])
        close   = _f(d['Close'].iloc[-1])
        prev_h  = _f(d['High'].iloc[-2])
        v_sma10 = _f(d['Volume'].rolling(10).mean().iloc[-1])
        def _rsi(s, n=14):
            delta = s.diff()
            gain  = delta.clip(lower=0).ewm(alpha=1/n, adjust=False).mean()
            loss  = (-delta.clip(upper=0)).ewm(alpha=1/n, adjust=False).mean()
            return 100 - (100 / (1 + gain / loss.replace(0, float('nan'))))
        rsi_val = _f(_rsi(d['Close'], 14).iloc[-1])
        return (close > ema100 and vol > 500_000 and close > 100
                and rsi_val > 55 and close > prev_h
                and vol > v_sma10 * 2)
    except: return False


def scan_btst_supertrend(d):
    """
    BTST Supertrend-based:
    close > open AND close > 1d ago close AND 1d ago close >= 2d ago close
    AND RSI(14) > 50 AND close > supertrend(7,3)
    AND high >= 52-week high
    Supertrend approximated using ATR.
    """
    try:
        if len(d) < 60: return False
        close  = d['Close']
        high   = d['High']
        low    = d['Low']
        # Supertrend(7,3) approximation
        atr = (high - low).rolling(7).mean() * 3
        upper_band = (high + low) / 2 + atr
        lower_band = (high + low) / 2 - atr
        st = lower_band.iloc[-1]  # simplified: use lower band as supertrend in uptrend
        c0 = _f(close.iloc[-1]); c1 = _f(close.iloc[-2]); c2 = _f(close.iloc[-3])
        o0 = _f(d['Open'].iloc[-1])
        # 52-week high
        wk52_high = _f(high.rolling(min(252, len(high))).max().iloc[-1])
        if not (c0 > o0 and c0 > c1 and c1 >= c2): return False
        if not (c0 > st): return False
        if not (_f(high.iloc[-1]) >= wk52_high * 0.99): return False
        def _rsi(s, n=14):
            delta = s.diff()
            gain  = delta.clip(lower=0).ewm(alpha=1/n, adjust=False).mean()
            loss  = (-delta.clip(upper=0)).ewm(alpha=1/n, adjust=False).mean()
            return 100 - (100 / (1 + gain / loss.replace(0, float('nan'))))
        return _f(_rsi(close, 14).iloc[-1]) > 50
    except: return False


def scan_trend_reversal(d):
    """
    Trend Reversal (Outside Bar):
    Today high > 1d ago high AND today low < 1d ago low (outside bar)
    AND close > open (bullish resolution)
    AND 1d ago close < 1d ago open (prior day was bearish)
    """
    try:
        if len(d) < 4: return False
        return (
            _f(d['High'].iloc[-1]) > _f(d['High'].iloc[-2]) and
            _f(d['Low'].iloc[-1])  < _f(d['Low'].iloc[-2])  and
            _f(d['Close'].iloc[-1]) > _f(d['Open'].iloc[-1]) and
            _f(d['Close'].iloc[-2]) < _f(d['Open'].iloc[-2])
        )
    except: return False


def scan_weekly_reversal_engulfing(d, w):
    """
    Weekly Bullish Engulfing:
    Weekly high > 1w ago high AND weekly low < 1w ago low
    AND weekly close > weekly open (this week bullish)
    AND 1w ago close < 1w ago open (prior week bearish)
    """
    try:
        if w is None or len(w) < 3: return False
        return (
            _f(w['High'].iloc[-1])  > _f(w['High'].iloc[-2]) and
            _f(w['Low'].iloc[-1])   < _f(w['Low'].iloc[-2])  and
            _f(w['Close'].iloc[-1]) > _f(w['Open'].iloc[-1]) and
            _f(w['Close'].iloc[-2]) < _f(w['Open'].iloc[-2])
        )
    except: return False


print("✅ New EOD scan functions ready: weekly_rsi_reversal, btst_strong, btst_supertrend, trend_reversal, weekly_reversal_engulfing")
now = datetime.datetime.now()

print(f" Running cell weekly RSI reveral -> Current date and time :{now} ")

## 💾 Supabase Config
Set your Supabase URL and Key below

In [ ]:

# ── SUPABASE CONFIG ───────────────────────────────────────────────────
# Set these once — get values from app.supabase.com → Settings → API
SUPABASE_URL = "https://YOUR_PROJECT.supabase.co"   # e.g. https://abcdefgh.supabase.co
SUPABASE_KEY = "YOUR_ANON_KEY"                       # Settings → API → anon public key

import os
SUPABASE_URL = os.environ.get("SUPABASE_URL", SUPABASE_URL)
SUPABASE_KEY = os.environ.get("SUPABASE_KEY", SUPABASE_KEY)

print(f"Supabase URL : {SUPABASE_URL[:30]}...")
print(f"Supabase Key : {SUPABASE_KEY[:20]}...")
now = datetime.datetime.now()

print(f" Running cell supabase config -> Current date and time :{now} ")

In [ ]:

# ── SUPABASE INSERT HELPER ────────────────────────────────────────────
import requests as _req, json as _json
from datetime import datetime as _dt

def db_insert(rows: list, scan_type: str, market: str):
    """
    Insert scan results into Supabase.
    rows: list of dicts with keys: scan_name, ticker, universe,
          close_price (opt), volume (opt), change_pct (opt)
    """
    if not rows:
        print(f"  db_insert: no rows for {scan_type}/{market}")
        return

    today = _dt.now().strftime('%Y-%m-%d')
    now   = _dt.now().isoformat()
    
    _market = os.environ.get("SCAN_MARKET", "usa").strip().lower()
    RUN_ID = os.environ.get("GITHUB_RUN_ID")

    # First delete today's rows for this scan_type + market (avoid duplicates on re-run)
    del_resp = _req.delete(
        f"{SUPABASE_URL}/rest/v1/scan_results",
        headers={
            "apikey":        SUPABASE_KEY,
            "Authorization": f"Bearer {SUPABASE_KEY}",
            "Content-Type":  "application/json",
        },
        params={
            "scan_date": f"eq.{today}",
            "scan_type": f"eq.{scan_type}",
            "market":    f"eq.{market}",
        }
    )
    if del_resp.status_code not in (200, 204):
        print(f"  ⚠ Delete existing rows: {del_resp.status_code} {del_resp.text[:80]}")

    # Build insert payload
    payload = []
    for r in rows:
        payload.append({
            "scan_date":   today,
            "scan_time":   now,
            "scan_type":   scan_type,
            "market":      market,
            "scanned_at":  datetime.now().isoformat(),
            "universe":    r.get("universe", ""),
            "scan_name":   r.get("scan_name", ""),
            "ticker":      r.get("ticker", ""),
            "close_price": r.get("close_price"),
            "volume":      r.get("volume"),
            "change_pct":  r.get("change_pct"),
            "avg_vol_20":  r.get("avg_vol_20"),
        })

    # Insert in batches of 500
    batch_size = 500
    total_inserted = 0
    for i in range(0, len(payload), batch_size):
        batch = payload[i:i+batch_size]
        resp = _req.post(
            f"{SUPABASE_URL}/rest/v1/scan_results",
            headers={
                "apikey":        SUPABASE_KEY,
                "Authorization": f"Bearer {SUPABASE_KEY}",
                "Content-Type":  "application/json",
                "Prefer":        "return=minimal",
            },
            data=_json.dumps(batch)
        )
        if resp.status_code in (200, 201):
            total_inserted += len(batch)
        else:
            print(f"  ❌ Insert batch {i//batch_size+1} failed: {resp.status_code} {resp.text[:100]}")

    print(f"  ✅ Inserted {total_inserted} rows → Supabase [{scan_type}/{market}]")
    return total_inserted
now = datetime.datetime.now()

print(f" Running cell supabase insert -> Current date and time :{now} ")

In [ ]:
# ============================================================
# EOD — JSON + EXCEL + SUPABASE
# ============================================================

import os
from pathlib import Path
from datetime import datetime
import pandas as pd

# ------------------------------------------------------------
# 1. Get the final EOD dataframe
# ------------------------------------------------------------
eod_df = df.copy()

# ------------------------------------------------------------
# 2. Output directory
# ------------------------------------------------------------
OUTDIR = Path("public/data")
OUTDIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 3. JSON
# ------------------------------------------------------------
eod_json = OUTDIR / "eod.json"

eod_df.to_json(
    eod_json,
    orient="records",
    indent=2,
    date_format="iso"
)

print(f"✅ EOD JSON created: {eod_json}")
print(f"   Rows: {len(eod_df)}")

# ------------------------------------------------------------
# 4. Excel
# ------------------------------------------------------------
eod_excel = OUTDIR / "eod.xlsx"

eod_df.to_excel(
    eod_excel,
    index=False
)

print(f"✅ EOD Excel created: {eod_excel}")

# ------------------------------------------------------------
# 5. Prepare rows for Supabase
# ------------------------------------------------------------

# Find ticker column
ticker_col = None

for col in ["Ticker", "ticker", "Symbol", "symbol", "Stock", "stock"]:
    if col in eod_df.columns:
        ticker_col = col
        break

if ticker_col is None:
    raise RuntimeError(
        f"Could not find ticker column. Available columns: {list(eod_df.columns)}"
    )

# Find scan/strategy column
scan_col = None

for col in ["Scan", "scan", "Scan_Name", "scan_name",
            "Strategy", "strategy", "Signal", "signal"]:
    if col in eod_df.columns:
        scan_col = col
        break

# If no scan column exists, use a fixed EOD name
if scan_col is None:
    eod_df["_scan_name"] = "EOD Scanner"
    scan_col = "_scan_name"

# Find universe column if available
universe_col = None

for col in ["Universe", "universe", "Market", "market"]:
    if col in eod_df.columns:
        universe_col = col
        break

# ------------------------------------------------------------
# 6. Convert to Supabase records
# ------------------------------------------------------------

supabase_rows = []

for _, row in eod_df.iterrows():

    ticker = str(row[ticker_col]).strip()

    if not ticker or ticker.lower() == "nan":
        continue

    scan_name = str(row[scan_col]).strip()

    if not scan_name or scan_name.lower() == "nan":
        scan_name = "EOD Scanner"

    if universe_col:
        universe = str(row[universe_col]).strip()
    else:
        universe = "EOD"

    if not universe or universe.lower() == "nan":
        universe = "EOD"

    supabase_rows.append({
        "scan_name": scan_name,
        "ticker": ticker,
        "universe": universe
    })

print(f"📊 Supabase rows prepared: {len(supabase_rows)}")

# ------------------------------------------------------------
# 7. Insert using the SAME writer used by Hourly
# ------------------------------------------------------------

from scripts.supabase_scan_writer import (
    upload_scan_records,
    cleanup_scan_history
)

uploaded = upload_scan_records(
    supabase_rows,
    market="eod",
    timeframe="eod",
    run_id=os.environ.get("GITHUB_RUN_ID")
)

print(f"✅ Supabase EOD rows uploaded: {uploaded}")

# ------------------------------------------------------------
# 8. Keep only 30 days of history
# ------------------------------------------------------------

cleanup_scan_history(30)

print("✅ EOD 30-day cleanup completed")
now = datetime.datetime.now()

print(f" Running cell json , excel, supabase -> Current date and time :{now} ")

In [ ]:
# DISABLED CELL
# # ── EOD: Insert all results into Supabase ────────────────────────────
# import os
# _market = os.environ.get("SCAN_MARKET", "both").upper()
# 
# def _build_rows(hits_list, scan_name, universe, daily_data=None):
#     rows = []
#     for t in hits_list:
#         row = {"scan_name": scan_name, "ticker": t, "universe": universe}
#         if daily_data and t in daily_data:
#             d = daily_data[t]
#             try:
#                 row["close_price"] = round(float(d["Close"].iloc[-1]), 2)
#                 row["volume"]      = int(d["Volume"].iloc[-1])
#                 row["change_pct"]  = round(((float(d["Close"].iloc[-1]) / float(d["Close"].iloc[-2])) - 1) * 100, 2)
#                 row["avg_vol_20"]  = int(d["Volume"].rolling(20).mean().iloc[-1])
#             except: pass
#         rows.append(row)
#     return rows
# 
# all_rows = []
# all_rows += _build_rows(sp500_hits,  "High Volume", "S&P 500")
# all_rows += _build_rows(r2000_hits,  "High Volume", "Russell 2000")
# all_rows += _build_rows(nifty_hits,  "High Volume", "Nifty 500")
# 
# db_insert(all_rows, scan_type="EOD", market=_market)
# print(f"EOD: {len(all_rows)} rows inserted")
